# Track B: five-year Alpaca/course-data alignment audit
AAPL, MSFT, NVDA, INTC, CSCO · July 1, 2020–June 30, 2025.

This notebook downloads **historical data only** and does not place orders or train models. It checks five-year coverage, stock-session bars, missing observations, causal index matching and availability of same-session four-hour outcomes. It does not certify execution prices or index identity.

**Run in Google Colab:** upload this notebook using File → Upload notebook, run the cells in order, and upload both course CSVs when prompted. Add `ALPACA_API_KEY` and `ALPACA_SECRET_KEY` to Colab Secrets (key icon) and enable notebook access, or use the hidden input prompts. Never paste credentials into code, chat, or GitHub. No credentials enter the exported report.

The course timestamp convention is unknown. Both `start` and `end` interpretations are tested as **hypothetical scenarios**. Neither is chosen by whichever produces better matching. Document the source convention before using index inputs for training.

In [2]:
%pip -q install pandas_market_calendars requests pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.1/141.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.3/213.3 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ibis-framework 9.5.0 requires toolz<1,>=0.11, but you have toolz 1.2.0 which is incompatible.


In [3]:
from pathlib import Path
import json, hashlib, time, getpass, shutil, sys, platform
import numpy as np
import pandas as pd
import requests
import pandas_market_calendars as mcal
from IPython.display import display

SYMBOLS = ['AAPL', 'MSFT', 'NVDA', 'INTC', 'CSCO']
START = '2020-07-01T04:00:00Z'  # NY midnight, inclusive
END = '2025-07-01T03:59:59Z'    # June 30 NY, inclusive
FEED = 'sip'   # Explicit; do not silently switch to IEX if access fails
ADJUSTMENT = 'split'  # Audit setting, not finalized model preprocessing
ASOF = '2025-06-30'
NY = 'America/New_York'
COURSE_TIMESTAMP_CONVENTION = None  # Set 'start'/'end' ONLY with source evidence
COURSE_METADATA_EVIDENCE = ''       # Source explanation/citation, not a guess
MAX_INDEX_AGE = pd.Timedelta('1h')   # Audit threshold, not an agreed strategy rule
ROOT = Path('track_b_data_check')
CACHE = ROOT / 'cache'
REPORTS = ROOT / 'reports'
CACHE.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)
COURSE_FILES = {
    'USA500': 'USA500.IDXUSD_1h_2020-07-01_to_2025-06-30_UTC.csv',
    'USATECH': 'USATECH.IDXUSD_1h_2020-07-01_to_2025-06-30_UTC.csv',
}

## 1. Upload course files and check their structure
These files contain overnight and non-equity-session data; do not manufacture stock bars to match them. The index volume definition and exact instrument identity need source documentation.

In [4]:
if not all(Path(p).exists() for p in COURSE_FILES.values()):
    from google.colab import files
    uploaded = files.upload()
    del uploaded
missing = [p for p in COURSE_FILES.values() if not Path(p).exists()]
if missing:
    raise FileNotFoundError('Upload these exact course filenames: ' + ', '.join(missing))

Saving USA500.IDXUSD_1h_2020-07-01_to_2025-06-30_UTC.csv to USA500.IDXUSD_1h_2020-07-01_to_2025-06-30_UTC.csv
Saving USATECH.IDXUSD_1h_2020-07-01_to_2025-06-30_UTC.csv to USATECH.IDXUSD_1h_2020-07-01_to_2025-06-30_UTC.csv


In [5]:
def normalize(df):
    required = ['timestamp', 'open', 'high', 'low', 'close', 'volume']
    if not set(required).issubset(df.columns):
        raise ValueError('Missing OHLCV/timestamp columns')
    d = df[required].copy()
    d['timestamp'] = pd.to_datetime(d['timestamp'], utc=True, errors='raise')
    for c in required[1:]:
        d[c] = pd.to_numeric(d[c], errors='raise')
    return d.sort_values('timestamp').reset_index(drop=True)

def structural_audit(name, d):
    px = d[['open', 'high', 'low', 'close']]
    bad_ohlc = (d.high < px.max(axis=1)) | (d.low > px.min(axis=1))
    return dict(series=name, rows=len(d), first_utc=str(d.timestamp.min()),
                last_utc=str(d.timestamp.max()), duplicates=int(d.timestamp.duplicated().sum()),
                missing_values=int(d.isna().sum().sum()),
                nonfinite_numeric=int((~np.isfinite(d.iloc[:, 1:])).sum().sum()),
                invalid_ohlc=int(bad_ohlc.sum()), nonpositive_prices=int((px<=0).any(axis=1).sum()),
                negative_volume=int((d.volume<0).sum()), zero_volume=int((d.volume==0).sum()))

course = {n: normalize(pd.read_csv(p)) for n,p in COURSE_FILES.items()}
course_audit = pd.DataFrame([structural_audit(n,d) for n,d in course.items()])
display(course_audit)
course_audit.to_csv(REPORTS/'course_structure.csv', index=False)
for n,d in course.items():
    if d.timestamp.duplicated().any():
        raise ValueError(f'{n}: duplicated timestamps must be resolved before alignment')
common = pd.Index(course['USA500'].timestamp).intersection(pd.Index(course['USATECH'].timestamp))
print('Common index timestamps:', len(common))

,series,rows,first_utc,last_utc,duplicates,missing_values,nonfinite_numeric,invalid_ohlc,nonpositive_prices,negative_volume,zero_volume
0,USA500,29510,2020-07-01 04:00:00+00:00,2025-07-01 04:00:00+00:00,0,0,0,0,0,0,0
1,USATECH,29508,2020-07-01 04:00:00+00:00,2025-07-01 04:00:00+00:00,0,0,0,0,0,0,0


Common index timestamps: 29507


## 2. Alpaca credentials and a small access check
The default requests the consolidated **SIP** feed. If access is refused, check your account/data permissions; this notebook does not purchase a plan or change feed automatically. IEX is a different feed and should be a coordinated, documented change.

In [6]:
def secret(name, prompt):
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = None
    return value or getpass.getpass(prompt)

API_KEY = secret('ALPACA_API_KEY', 'Alpaca API key (hidden): ')
API_SECRET = secret('ALPACA_SECRET_KEY', 'Alpaca secret key (hidden): ')
if not API_KEY or not API_SECRET:
    raise ValueError('Both Alpaca credentials are required')
SESSION = requests.Session()
SESSION.headers.update({'APCA-API-KEY-ID': API_KEY, 'APCA-API-SECRET-KEY': API_SECRET})
del API_KEY, API_SECRET

def fetch_symbol(symbol, start, end):
    params = dict(timeframe='1Hour', start=start, end=end, feed=FEED,
                  adjustment=ADJUSTMENT, asof=ASOF, limit=10000, sort='asc')
    request_record = dict(symbol=symbol, **params)
    digest = hashlib.sha256(json.dumps(request_record, sort_keys=True).encode()).hexdigest()[:16]
    path = CACHE/f'{symbol}_{digest}.csv'
    if path.exists():
        return normalize(pd.read_csv(path))
    rows, token, seen_tokens = [], None, set()
    while True:
        query = dict(params)
        if token:
            query['page_token'] = token
        for attempt in range(6):
            response = SESSION.get(f'https://data.alpaca.markets/v2/stocks/{symbol}/bars',
                                   params=query, timeout=90)
            if response.status_code in (429,500,502,503,504):
                time.sleep(min(2**attempt, 30))
                continue
            break
        if response.status_code != 200:
            raise RuntimeError(f'{symbol}: Alpaca returned HTTP {response.status_code}. '
                               'Check credentials, feed permissions and parameters. '
                               'No alternate feed was selected.')
        payload = response.json()
        rows.extend(payload.get('bars') or [])
        token = payload.get('next_page_token')
        if not token:
            break
        if token in seen_tokens:
            raise RuntimeError('Repeated pagination token: download stopped')
        seen_tokens.add(token)
    if not rows:
        raise RuntimeError(f'{symbol}: no bars returned for {start}–{end}')
    d = pd.DataFrame(rows).rename(columns=dict(t='timestamp',o='open',h='high',l='low',c='close',v='volume'))
    d = normalize(d)
    if d.timestamp.duplicated().any():
        raise RuntimeError(f'{symbol}: duplicate bars returned; inspect before continuing')
    d.to_csv(path,index=False)
    return d

sample = fetch_symbol('AAPL', '2024-07-08T04:00:00Z', '2024-07-13T03:59:59Z')
display(sample.head())
print('Access check passed:', len(sample), 'bars; feed:', FEED)

Alpaca API key (hidden): ··········
Alpaca secret key (hidden): ··········


,timestamp,open,high,low,close,volume
0,2024-07-08 08:00:00+00:00,227.60,227.60,226.610,227.50,61379
1,2024-07-08 09:00:00+00:00,227.36,227.55,227.130,227.16,21446
2,2024-07-08 10:00:00+00:00,227.23,227.27,227.070,227.10,15437
3,2024-07-08 11:00:00+00:00,227.10,227.88,226.880,227.84,108457
4,2024-07-08 12:00:00+00:00,227.00,228.21,225.775,227.48,662545


Access check passed: 80 bars; feed: sip


## 3. Download all five stock histories
All symbols use the same feed and adjustment. Successful downloads are cached so rerunning the notebook within this runtime does not repeat them. Colab runtime files are temporary; download the audit bundle at the end.

In [7]:
stocks = {}
for symbol in SYMBOLS:
    stocks[symbol] = fetch_symbol(symbol, START, END)
    print(symbol, len(stocks[symbol]), 'bars downloaded/cached')
stock_audit = pd.DataFrame([structural_audit(s,d) for s,d in stocks.items()])
display(stock_audit)
stock_audit.to_csv(REPORTS/'stock_structure.csv', index=False)

AAPL 19768 bars downloaded/cached
MSFT 19752 bars downloaded/cached
NVDA 19667 bars downloaded/cached
INTC 19726 bars downloaded/cached
CSCO 18056 bars downloaded/cached


,series,rows,first_utc,last_utc,duplicates,missing_values,nonfinite_numeric,invalid_ohlc,nonpositive_prices,negative_volume,zero_volume
0,AAPL,19768,2020-07-01 08:00:00+00:00,2025-06-30 23:00:00+00:00,0,0,0,0,0,0,0
1,MSFT,19752,2020-07-01 08:00:00+00:00,2025-06-30 23:00:00+00:00,0,0,0,0,0,0,0
2,NVDA,19667,2020-07-01 10:00:00+00:00,2025-06-30 23:00:00+00:00,0,0,0,0,0,0,0
3,INTC,19726,2020-07-01 08:00:00+00:00,2025-06-30 23:00:00+00:00,0,0,0,0,0,0,0
4,CSCO,18056,2020-07-01 09:00:00+00:00,2025-06-30 23:00:00+00:00,0,0,0,0,0,0,0


## 4. Expected regular-session full-hour bars
Alpaca's 1Hour bars aggregate minute bars within the clock hour. Treat timestamps as interval starts. Use an NYSE session calendar with holidays and early closes. Keep only whole intervals fully inside the session: on a normal day 10:00–11:00 through 15:00–16:00 New York. Exclude the 09:00–10:00 opening interval because part of it is outside the regular session. A real model can later use minute data to construct opening-aligned bars; this audit does not do so.

In [8]:
def expected_grid():
    schedule = mcal.get_calendar('NYSE').schedule(start_date='2020-07-01', end_date='2025-06-30')
    rows=[]
    for day, session in schedule.iterrows():
        op, cl = session.market_open, session.market_close
        for start in pd.date_range(op.ceil('h'), cl, freq='h'):
            end = start + pd.Timedelta('1h')
            if end <= cl:
                rows.append(dict(session=str(day.date()), bar_start=start, bar_end=end,
                                 session_open=op, session_close=cl))
    return pd.DataFrame(rows)

grid = expected_grid()
print('Expected full regular-session hours:', len(grid))
panel = grid.copy()
coverage=[]
for symbol,d in stocks.items():
    present = grid.bar_start.isin(d.timestamp)
    panel[symbol] = present
    year = grid.bar_start.dt.tz_convert(NY).dt.year
    for y in sorted(year.unique()):
        m = year.eq(y)
        coverage.append(dict(symbol=symbol,year=int(y),expected=int(m.sum()),
                             observed=int(present[m].sum()),missing=int((~present[m]).sum()),
                             coverage_pct=round(100*present[m].mean(),3)))
    grid.loc[~present,['session','bar_start','bar_end']].to_csv(REPORTS/f'{symbol}_missing_hours.csv',index=False)
coverage=pd.DataFrame(coverage)
coverage.to_csv(REPORTS/'stock_yearly_coverage.csv',index=False)
display(coverage)
panel['all_five_present']=panel[SYMBOLS].all(axis=1)
print('Hours with all five stocks:', int(panel.all_five_present.sum()), '/',len(panel))

Expected full regular-session hours: 7503


,symbol,year,expected,observed,missing,coverage_pct
0,AAPL,2020,762,762,0,100.0
1,AAPL,2021,1509,1509,0,100.0
2,AAPL,2022,1503,1503,0,100.0
3,AAPL,2023,1494,1494,0,100.0
4,AAPL,2024,1503,1503,0,100.0
5,AAPL,2025,732,732,0,100.0
6,MSFT,2020,762,762,0,100.0
7,MSFT,2021,1509,1509,0,100.0
8,MSFT,2022,1503,1503,0,100.0
9,MSFT,2023,1494,1494,0,100.0


Hours with all five stocks: 7503 / 7503


## 5. Causal course-index matching under both timestamp hypotheses
A course bar is usable only after it ends. Match backward to the latest completed course bar **within the same stock session**, and permit at most one hour of age as an audit scenario. No prior-day or overnight forward-filling. This checks availability and lag; it does not establish instrument equivalence, economic validity or optimal features.

In [9]:
def match_index(grid, d, convention):
    q = d[['timestamp','close']].copy()
    q['index_start'] = q.timestamp if convention=='start' else q.timestamp-pd.Timedelta('1h')
    q['index_end'] = q.index_start+pd.Timedelta('1h')
    matched = pd.merge_asof(grid.sort_values('bar_end'),q.sort_values('index_end'),
                            left_on='bar_end',right_on='index_end',direction='backward',
                            tolerance=MAX_INDEX_AGE)
    matched['age_minutes']=(matched.bar_end-matched.index_end).dt.total_seconds()/60
    matched['available']=matched.index_end.notna() & (matched.index_end<=matched.bar_end) & (matched.index_end>matched.session_open)
    assert not (matched.loc[matched.available,'index_end']>matched.loc[matched.available,'bar_end']).any()
    return matched

scenarios = ['start','end'] if COURSE_TIMESTAMP_CONVENTION is None else [COURSE_TIMESTAMP_CONVENTION]
if any(s not in ['start','end'] for s in scenarios):
    raise ValueError('Timestamp convention must be None, start, or end')
if COURSE_TIMESTAMP_CONVENTION is not None and not COURSE_METADATA_EVIDENCE.strip():
    raise ValueError('Document the source evidence before setting the convention')
alignment=[]
for convention in scenarios:
    working=panel.copy()
    for name,d in course.items():
        matched=match_index(grid,d,convention)
        working[name+'_available']=matched.available.to_numpy()
        matched.to_csv(REPORTS/f'{name}_{convention}_matches.csv',index=False)
        for y in sorted(grid.bar_start.dt.tz_convert(NY).dt.year.unique()):
            m=grid.bar_start.dt.tz_convert(NY).dt.year.eq(y)
            ages=matched.loc[m & matched.available,'age_minutes']
            alignment.append(dict(scenario=convention,index=name,year=int(y),
                                  expected=int(m.sum()),matched=int(matched.loc[m,'available'].sum()),
                                  matched_pct=round(100*matched.loc[m,'available'].mean(),3),
                                  maximum_age_minutes=ages.max()))
    working['all_inputs_available']=working.all_five_present & working.USA500_available & working.USATECH_available
    # Require every intermediate hourly stock bar for a continuous four-hour outcome.
    working['four_hour_stock_window']=working.all_five_present.copy()
    for step in range(1,5):
        future=working.groupby('session')['all_five_present'].shift(-step).fillna(False).astype(bool)
        working['four_hour_stock_window'] &= future
    working['candidate_origins']=working.all_inputs_available & working.four_hour_stock_window
    working.to_csv(REPORTS/f'panel_{convention}_eligibility.csv',index=False)
    print(convention, 'hypothesis:', int(working.all_inputs_available.sum()),'input rows;',
          int(working.candidate_origins.sum()),'same-session four-hour candidate origins')
    print('Candidates are timing feasibility only, before lookback requirements or executable entry/exit choices.')
alignment=pd.DataFrame(alignment)
display(alignment)
alignment.to_csv(REPORTS/'index_yearly_alignment.csv',index=False)

/tmp/ipykernel_801/3239791967.py:36: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  future=working.groupby('session')['all_five_present'].shift(-step).fillna(False).astype(bool)
/tmp/ipykernel_801/3239791967.py:36: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  future=working.groupby('session')['all_five_present'].shift(-step).fillna(False).astype(bool)
/tmp/ipykernel_801/3239791967.py:36: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instea

start hypothesis: 7503 input rows; 2492 same-session four-hour candidate origins
Candidates are timing feasibility only, before lookback requirements or executable entry/exit choices.


/tmp/ipykernel_801/3239791967.py:36: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  future=working.groupby('session')['all_five_present'].shift(-step).fillna(False).astype(bool)
/tmp/ipykernel_801/3239791967.py:36: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  future=working.groupby('session')['all_five_present'].shift(-step).fillna(False).astype(bool)
/tmp/ipykernel_801/3239791967.py:36: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instea

end hypothesis: 7503 input rows; 2492 same-session four-hour candidate origins
Candidates are timing feasibility only, before lookback requirements or executable entry/exit choices.


,scenario,index,year,expected,matched,matched_pct,maximum_age_minutes
0,start,USA500,2020,762,762,100.0,0.0
1,start,USA500,2021,1509,1509,100.0,0.0
2,start,USA500,2022,1503,1503,100.0,0.0
3,start,USA500,2023,1494,1494,100.0,0.0
4,start,USA500,2024,1503,1503,100.0,60.0
5,start,USA500,2025,732,732,100.0,0.0
6,start,USATECH,2020,762,762,100.0,0.0
7,start,USATECH,2021,1509,1509,100.0,0.0
8,start,USATECH,2022,1503,1503,100.0,0.0
9,start,USATECH,2023,1494,1494,100.0,0.0


## 6. Export findings
Inspect coverage by **each year**, not only total rows. Compare winter/summer matched timestamps and missing-hour files. Missing values must not be filled with future observations. An all-five intersection is an audit subset, not a finalized point-in-time eligibility rule. Do not select stocks or the timestamp scenario using model/test performance.

**Still required before training:** course-bar timestamp evidence, instrument/volume definitions, agreed execution timing, outcome/benchmark definition, causal lookback features and training-only scaling, chronological splits and target purging. This notebook intentionally does not create predictions or backtest profits.

In [ ]:
metadata=dict(symbols=SYMBOLS,start=START,end=END,feed=FEED,adjustment=ADJUSTMENT,asof=ASOF,
              course_timestamp_convention=COURSE_TIMESTAMP_CONVENTION,
              course_metadata_evidence=COURSE_METADATA_EVIDENCE,
              status='PROVISIONAL: course timestamp evidence required' if COURSE_TIMESTAMP_CONVENTION is None else 'Timing audit only; remaining modeling decisions unresolved',
              created_utc=str(pd.Timestamp.now(tz='UTC')),python=platform.python_version(),
              pandas=pd.__version__,calendar_library=mcal.__version__,
              maximum_index_age_minutes=MAX_INDEX_AGE.total_seconds()/60,
              source_hashes={n:hashlib.sha256(Path(p).read_bytes()).hexdigest() for n,p in COURSE_FILES.items()})
(REPORTS/'audit_metadata.json').write_text(json.dumps(metadata,indent=2))
(REPORTS/'README.txt').write_text('Coverage and causal alignment audit only. No model was trained.\nCourse start/end conventions are hypotheses until documented.\nFour-hour candidate origins do not certify executable entry/exit prices.\nInspect yearly coverage and missing-hour files.\n')
archive=shutil.make_archive('Track_B_Alpaca_Audit_Results','zip',REPORTS)
print('Report bundle:',archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass

Report bundle: /content/Track_B_Alpaca_Audit_Results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.upload()

Saving Track_B_Quick_Experiment.py to Track_B_Quick_Experiment.py


{'Track_B_Quick_Experiment.py': b'"""Run inside the audit notebook with: %run -i Track_B_Quick_Experiment.py\n\nUses existing stocks, course and grid variables; does not download or place trades.\nStart-of-bar semantics and chronological split dates remain provisional.\n"""\nfrom pathlib import Path\nimport json\nimport numpy as np\nimport pandas as pd\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.linear_model import Ridge\nfrom sklearn.metrics import mean_absolute_error, mean_squared_error\n\nneeded = [\'stocks\', \'course\', \'grid\']\nif any(name not in globals() for name in needed):\n    raise RuntimeError(\'Run sections 1\xe2\x80\x934 of the Alpaca audit notebook in this runtime first. \'\n                       \'This cell needs its stocks, course and grid variables.\')\n\nSYMBOLS_QUICK = [\'AAPL\', \'MSFT\', \'NVDA\', \'INTC\', \'CSCO\']\n# Replace only if you and Sumeet have already agreed on different boundaries.\nT

In [ ]:
%run -i Track_B_Quick_Experiment.py

PROVISIONAL pilot: course timestamps assumed to mark interval START.
Training: July 2020–December 2022; validation: calendar 2023.
January 2024 onward is not used for fitting, tuning or scoring.
Market arms include fixed momentum/volatility interactions with market features.
Training origins: 1238 | validation origins: 496

Lower MAE/RMSE is better; higher mean rank IC is better. Errors are log-return units.
                   model  features      MAE     RMSE  mean_rank_IC  scored_origins  MAE_change_vs_stock_only  IC_change_vs_stock_only
              Stock only         7 0.004221 0.005717      0.010484             496                  0.000000                 0.000000
          Stock + USA500        19 0.004216 0.005719      0.043750             496                 -0.000004                 0.033266
Stock + USA500 + USATECH        31 0.004226 0.005734      0.054234             496                  0.000005                 0.043750

Saved: /content/Track_B_Quick_Experiment_Results.zi

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.upload()  # Select Track_B_LSTM_Pilot.py

Saving Track_B_LSTM_Pilot.py to Track_B_LSTM_Pilot.py


{'Track_B_LSTM_Pilot.py': b'"""Exploratory shared-LSTM pilot. Run in existing Colab: %run -i Track_B_LSTM_Pilot.py\nRequires stocks, course and grid from the audit notebook. No downloads or trades.\nStart-of-bar timing and all modeling settings remain provisional.\n"""\nfrom pathlib import Path\nimport json, random, shutil, time\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.utils.data import DataLoader, TensorDataset\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.metrics import mean_absolute_error, mean_squared_error\n\nif any(name not in globals() for name in [\'stocks\',\'course\',\'grid\']):\n    raise RuntimeError(\'First run sections 1\xe2\x80\x934 of the audit notebook in this Colab runtime.\')\n\nNAMES = [\'AAPL\',\'MSFT\',\'NVDA\',\'INTC\',\'CSCO\']\nLOOKBACK = 48\nHIDDEN = 32\nSEED = 42\nMAX_EPOCHS = 12\nPATIENCE = 3\nBATCH = 128\nLEARNING_RATE = 0.001\nINNER_END = pd.Timestamp(\'2022-10-01\',tz=\'UTC\')\nTRAIN_END = 

In [ ]:
%run -i Track_B_LSTM_Pilot.py

PROVISIONAL LSTM PILOT | start-of-bar assumption | device: cpu
One shared model per arm, 48 observations, hidden width 32, one seed.
2022 Q4 selects epoch count; fresh refit through 2022; 2023 evaluates once.
2024 onward is not fitted or scored.
Common train origins: 1222 | common 2023 origins: 496
LSTM stock only selection epoch 1 inner MSE 0.81522
LSTM stock only selection epoch 2 inner MSE 0.81486
LSTM stock only selection epoch 3 inner MSE 0.81792
LSTM stock only selection epoch 4 inner MSE 0.81578
LSTM stock only selection epoch 5 inner MSE 0.81719
LSTM stock only refit epoch 1 / 2
LSTM stock only refit epoch 2 / 2
LSTM stock + USA500 selection epoch 1 inner MSE 0.81564
LSTM stock + USA500 selection epoch 2 inner MSE 0.81483
LSTM stock + USA500 selection epoch 3 inner MSE 0.8123
LSTM stock + USA500 selection epoch 4 inner MSE 0.81319
LSTM stock + USA500 selection epoch 5 inner MSE 0.8119
LSTM stock + USA500 selection epoch 6 inner MSE 0.81291
LSTM stock + USA500 selection epoch 7 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Sources
- Alpaca historical bars: https://docs.alpaca.markets/us/reference/stockbars — pagination, feeds, adjustments and authentication.
- Alpaca Market Data FAQ: https://docs.alpaca.markets/us/docs/market-data-faq — bar aggregation and timestamp conventions.
- Calendar package: https://pandas-market-calendars.readthedocs.io/ — exchange sessions, holidays and early closes.

No authenticated Alpaca download was run when this notebook was prepared. Local verification covers course files and synthetic timing/coverage cases only.